In [1]:
from pathlib import Path
import json
import joblib
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
PROJECT_DIR = Path.cwd()

REPORTS_DIR = PROJECT_DIR / "reports"
MODELS_DIR = PROJECT_DIR / "models"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Project folder:", PROJECT_DIR)
print("Reports folder:", REPORTS_DIR)
print("Models folder:", MODELS_DIR)

Project folder: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation
Reports folder: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\reports
Models folder: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\models


In [3]:
dataset = load_breast_cancer(as_frame=True)

X = dataset.data
y = dataset.target

FEATURE_COLUMNS = X.columns.tolist()

print("Dataset shape:", X.shape)
print("Number of features:", len(FEATURE_COLUMNS))
print("\nTarget distribution:")
print(y.value_counts().sort_index())

print("\nTarget meanings:")
print("0 = Malignant")
print("1 = Benign")

Dataset shape: (569, 30)
Number of features: 30

Target distribution:
target
0    212
1    357
Name: count, dtype: int64

Target meanings:
0 = Malignant
1 = Benign


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

assert set(X_train.index).isdisjoint(set(X_test.index))
assert len(X_train) + len(X_test) == len(X)

print("Train/test split validation: PASSED")

Training records: 455
Testing records: 114
Train/test split validation: PASSED


In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            FEATURE_COLUMNS
        )
    ],
    remainder="drop"
)

print("Preprocessing pipeline created successfully")

Preprocessing pipeline created successfully


In [6]:
models = {
    "Dummy Classifier": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", DummyClassifier(strategy="most_frequent"))
    ]),

    "Logistic Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000))
    ]),

    "Random Forest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=RANDOM_STATE,
            class_weight="balanced"
        ))
    ])
}

print("Models to compare:")
for model_name in models:
    print("-", model_name)

Models to compare:
- Dummy Classifier
- Logistic Regression
- Random Forest


In [7]:
results = []
trained_models = {}
predictions = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    trained_models[model_name] = model
    predictions[model_name] = y_pred

    results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "malignant_precision": precision_score(
            y_test, y_pred, pos_label=0, zero_division=0
        ),
        "malignant_recall": recall_score(
            y_test, y_pred, pos_label=0, zero_division=0
        ),
        "malignant_f1": f1_score(
            y_test, y_pred, pos_label=0, zero_division=0
        ),
        "benign_recall": recall_score(
            y_test, y_pred, pos_label=1, zero_division=0
        )
    })

print("All three models trained and evaluated.")

All three models trained and evaluated.


In [8]:
metrics_df = pd.DataFrame(results)

metrics_df = metrics_df.sort_values(
    by=["malignant_recall", "malignant_f1", "accuracy"],
    ascending=False
).reset_index(drop=True)

print(metrics_df.round(4).to_string(index=False))

metrics_path = REPORTS_DIR / "day3_model_comparison.csv"
metrics_df.to_csv(metrics_path, index=False)

print("\nMetrics saved to:", metrics_path)

              model  accuracy  malignant_precision  malignant_recall  malignant_f1  benign_recall
Logistic Regression    0.9825               0.9762            0.9762        0.9762         0.9861
      Random Forest    0.9474               0.9286            0.9286        0.9286         0.9583
   Dummy Classifier    0.6316               0.0000            0.0000        0.0000         1.0000

Metrics saved to: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\reports\day3_model_comparison.csv


In [9]:
dummy_row = metrics_df.loc[
    metrics_df["model"] == "Dummy Classifier"
].iloc[0]

comparison_df = metrics_df.copy()

for metric in [
    "accuracy",
    "malignant_precision",
    "malignant_recall",
    "malignant_f1",
    "benign_recall"
]:
    comparison_df[f"{metric}_improvement_vs_dummy"] = (
        comparison_df[metric] - dummy_row[metric]
    )

comparison_path = REPORTS_DIR / "day3_baseline_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print(comparison_df.round(4).to_string(index=False))
print("\nBaseline comparison saved to:", comparison_path)

              model  accuracy  malignant_precision  malignant_recall  malignant_f1  benign_recall  accuracy_improvement_vs_dummy  malignant_precision_improvement_vs_dummy  malignant_recall_improvement_vs_dummy  malignant_f1_improvement_vs_dummy  benign_recall_improvement_vs_dummy
Logistic Regression    0.9825               0.9762            0.9762        0.9762         0.9861                         0.3509                                    0.9762                                 0.9762                             0.9762                             -0.0139
      Random Forest    0.9474               0.9286            0.9286        0.9286         0.9583                         0.3158                                    0.9286                                 0.9286                             0.9286                             -0.0417
   Dummy Classifier    0.6316               0.0000            0.0000        0.0000         1.0000                         0.0000                             

In [10]:
for model_name, y_pred in predictions.items():
    print("\n" + "=" * 55)
    print(model_name)
    print("=" * 55)

    print("Confusion matrix (rows = actual, columns = predicted):")
    print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

    print("\nClassification report:")
    print(classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["Malignant", "Benign"],
        zero_division=0
    ))


Dummy Classifier
Confusion matrix (rows = actual, columns = predicted):
[[ 0 42]
 [ 0 72]]

Classification report:
              precision    recall  f1-score   support

   Malignant       0.00      0.00      0.00        42
      Benign       0.63      1.00      0.77        72

    accuracy                           0.63       114
   macro avg       0.32      0.50      0.39       114
weighted avg       0.40      0.63      0.49       114


Logistic Regression
Confusion matrix (rows = actual, columns = predicted):
[[41  1]
 [ 1 71]]

Classification report:
              precision    recall  f1-score   support

   Malignant       0.98      0.98      0.98        42
      Benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


Random Forest
Confusion matrix (rows = actual, columns = predicted):
[[39  3]
 [ 3 69]]

Classification report

In [11]:
selected_row = metrics_df.iloc[0]
selected_model_name = selected_row["model"]

selected_model = trained_models[selected_model_name]

print("Selected candidate:", selected_model_name)
print("Malignant recall:", round(
    selected_row["malignant_recall"], 4
))
print("Malignant F1:", round(
    selected_row["malignant_f1"], 4
))
print("Accuracy:", round(
    selected_row["accuracy"], 4
))

if selected_model_name == "Dummy Classifier":
    print("WARNING: The selected candidate is only a dummy baseline.")
    print("Review the comparison before considering this a useful model.")

model_path = MODELS_DIR / "day3_selected_model.joblib"
joblib.dump(selected_model, model_path)

selection_path = REPORTS_DIR / "day3_model_selection.json"

with open(selection_path, "w", encoding="utf-8") as file:
    json.dump({
        "selected_model": selected_model_name,
        "selection_rule": (
            "Highest malignant recall, then malignant F1, "
            "then accuracy"
        ),
        "random_state": RANDOM_STATE,
        "test_size": TEST_SIZE,
        "malignant_class_label": 0,
        "metrics": {
            key: float(selected_row[key])
            for key in [
                "accuracy",
                "malignant_precision",
                "malignant_recall",
                "malignant_f1",
                "benign_recall"
            ]
        }
    }, file, indent=4)

print("Saved model:", model_path)
print("Saved selection record:", selection_path)

Selected candidate: Logistic Regression
Malignant recall: 0.9762
Malignant F1: 0.9762
Accuracy: 0.9825
Saved model: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\models\day3_selected_model.joblib
Saved selection record: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\reports\day3_model_selection.json


In [12]:
def validate_input(data):
    if not isinstance(data, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if data.empty:
        raise ValueError("Input must contain at least one row.")

    if data.columns.duplicated().any():
        raise ValueError("Duplicate feature columns are not allowed.")

    missing = set(FEATURE_COLUMNS) - set(data.columns)
    extra = set(data.columns) - set(FEATURE_COLUMNS)

    if missing:
        raise ValueError(f"Missing required features: {sorted(missing)}")

    if extra:
        raise ValueError(f"Unexpected features: {sorted(extra)}")

    if not all(
        pd.api.types.is_numeric_dtype(data[col])
        for col in FEATURE_COLUMNS
    ):
        raise TypeError("All features must be numeric.")

    return data[FEATURE_COLUMNS]


test_results = []

# Positive case: valid input
valid_input = validate_input(X_test.iloc[[0]])
valid_prediction = selected_model.predict(valid_input)

test_results.append({
    "test": "Valid input",
    "expected": "Prediction returned",
    "actual": str(valid_prediction[0]),
    "status": "PASSED"
})

# Boundary case: one row
single_row = validate_input(X_test.iloc[[0]])
assert len(selected_model.predict(single_row)) == 1

test_results.append({
    "test": "Single-row input",
    "expected": "One prediction",
    "actual": "One prediction",
    "status": "PASSED"
})

# Negative case: missing required feature
try:
    invalid_input = X_test.iloc[[0]].drop(columns=[FEATURE_COLUMNS[0]])
    validate_input(invalid_input)
    raise AssertionError("Missing feature was not rejected.")
except ValueError as error:
    if "Missing required features" not in str(error):
        raise
    print("Missing-feature test: PASSED (rejected)")

    test_results.append({
        "test": "Missing feature",
        "expected": "Input rejected",
        "actual": "Input rejected",
        "status": "PASSED"
    })

# Negative case: empty input
try:
    validate_input(X_test.iloc[0:0])
    raise AssertionError("Empty input was not rejected.")
except ValueError as error:
    if "at least one row" not in str(error):
        raise
    print("Empty-input test: PASSED (rejected)")

    test_results.append({
        "test": "Empty input",
        "expected": "Input rejected",
        "actual": "Input rejected",
        "status": "PASSED"
    })

tests_df = pd.DataFrame(test_results)
tests_path = REPORTS_DIR / "day3_inference_tests.csv"
tests_df.to_csv(tests_path, index=False)

print(tests_df.to_string(index=False))
print("\nInference tests saved to:", tests_path)

Missing-feature test: PASSED (rejected)
Empty-input test: PASSED (rejected)
            test            expected         actual status
     Valid input Prediction returned              0 PASSED
Single-row input      One prediction One prediction PASSED
 Missing feature      Input rejected Input rejected PASSED
     Empty input      Input rejected Input rejected PASSED

Inference tests saved to: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\reports\day3_inference_tests.csv


In [13]:
loaded_model = joblib.load(model_path)

original_prediction = selected_model.predict(X_test.iloc[[0]])
reloaded_prediction = loaded_model.predict(X_test.iloc[[0]])

assert (original_prediction == reloaded_prediction).all()

print("Saved model reload test: PASSED")
print("Prediction before reload:", original_prediction[0])
print("Prediction after reload:", reloaded_prediction[0])

Saved model reload test: PASSED
Prediction before reload: 0
Prediction after reload: 0


In [15]:
best = metrics_df.iloc[0]

report_text = f"""# Day 3 — Baseline Training and Evaluation

**Date:** Wednesday, 21 October 2026  
**Story:** Baseline training and evaluation  
**Project:** Testable ML Training and Inference Pipeline

## 1. Objective

Train at least two candidate models, compare their metrics against a Dummy Classifier, and justify a baseline selection.

## 2. Dataset and evaluation setup

- Dataset: Breast Cancer Wisconsin Diagnostic (scikit-learn)
- Records: {len(X)}
- Features: {len(FEATURE_COLUMNS)} numeric features
- Target: `target` (0 = malignant, 1 = benign)
- Training records: {len(X_train)}
- Testing records: {len(X_test)}
- Split: stratified 80/20
- Random state: {RANDOM_STATE}

## 3. Candidate models

1. Dummy Classifier (`most_frequent`)
2. Logistic Regression
3. Random Forest (`n_estimators=200`, `random_state=42`,
   `class_weight=balanced`)

## 4. Preprocessing and leakage prevention

Median imputation and standard scaling are inside a scikit-learn
Pipeline and ColumnTransformer. The complete pipeline is fitted only
on the training data and then evaluated on the held-out test data.

## 5. Evaluation metrics

Metrics include accuracy, malignant precision, malignant recall,
malignant F1-score, and benign recall. Malignant recall receives
particular attention because class 0 represents malignant cases.

See `reports/day3_model_comparison.csv` and
`reports/day3_baseline_comparison.csv` for the measured results.

## 6. Selection rule and result

Selection rule: highest malignant recall, followed by malignant F1,
then accuracy.

Selected candidate: **{selected_model_name}**

- Accuracy: {best['accuracy']:.4f}
- Malignant precision: {best['malignant_precision']:.4f}
- Malignant recall: {best['malignant_recall']:.4f}
- Malignant F1-score: {best['malignant_f1']:.4f}
- Benign recall: {best['benign_recall']:.4f}

The selection must also be compared with the Dummy Classifier results.
This model is an educational baseline, not a clinically validated
diagnostic system.

## 7. Validation and evidence

- Valid input prediction tested.
- Single-row boundary case tested.
- Missing required feature rejected.
- Empty input rejected.
- Saved model reloaded and prediction consistency checked.

Detailed results are stored in `reports/day3_inference_tests.csv`.

## 8. Execution steps

1. Open `day3_baseline_evaluation.ipynb` in VS Code.
2. Run the notebook cells from top to bottom.
3. Review the metric table and confusion matrices.
4. Confirm the inference tests and model reload test pass.
5. Review the generated files and commit the changes to Git.

## 9. Known limitations

- Results depend on this dataset and the fixed train/test split.
- Only three candidate models were compared.
- No external validation dataset or clinical validation was used.
- The selection rule prioritizes malignant recall and may not reflect
  every real-world cost or deployment requirement.
- Results should not be interpreted as medical advice or a clinical tool.

## 10. Completion checklist

- [ ] Notebook runs successfully.
- [ ] Three candidate models compared with the Dummy Classifier.
- [ ] Metric comparison CSV generated.
- [ ] Negative/boundary tests passed.
- [ ] Selected model saved and reload verified.
- [ ] Documentation reviewed.
- [ ] Changes committed to Git.
"""

notes_path = PROJECT_DIR / "DAY3_BASELINE_NOTES.md"
notes_path.write_text(report_text, encoding="utf-8")

print("Day 3 notes saved to:", notes_path)

Day 3 notes saved to: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\DAY3_BASELINE_NOTES.md


In [16]:
assert len(X_train) == 455
assert len(X_test) == 114
assert len(models) >= 3
assert metrics_df["model"].nunique() >= 3
assert metrics_path.exists()
assert comparison_path.exists()
assert selection_path.exists()
assert model_path.exists()
assert tests_path.exists()
assert notes_path.exists()

assert (tests_df["status"] == "PASSED").all()

print("Day 3 acceptance checks: PASSED")
print("\nGenerated files:")
for path in [
    PROJECT_DIR / "day3_baseline_evaluation.ipynb",
    metrics_path,
    comparison_path,
    selection_path,
    tests_path,
    model_path,
    notes_path
]:
    print("-", path, "| exists:", path.exists())

Day 3 acceptance checks: PASSED

Generated files:
- c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\day3_baseline_evaluation.ipynb | exists: True
- c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\reports\day3_model_comparison.csv | exists: True
- c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\reports\day3_baseline_comparison.csv | exists: True
- c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\reports\day3_model_selection.json | exists: True
- c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\reports\day3_inference_tests.csv | exists: True
- c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\models\day3_selected_model.joblib | exists: True
- c:\Users\Work\Deskt